# 14.04 · QLoRA on Colab, then serve it locally

**Module 14 · Local Models & Fine-Tuning** · Part 2 (Pro) · elective

| | |
|---|---|
| ⏱️ Estimated time | 3–4 h (about 15 min of it on a Colab GPU) |
| 📋 Prerequisites | Lesson 14.03 (dataset, baselines, paired comparisons) and 14.01 (Ollama). |
| 🔑 Keys | none · a Google account for Colab · optional `HF_TOKEN` to push your model to the Hub |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Explain** LoRA and QLoRA: what is trained, what is frozen, and why it fits a free GPU.
2. **Estimate** the GPU memory a QLoRA run needs, and know why a CPU-only laptop trains on Colab instead.
3. **Run** a QLoRA fine-tune of Qwen3 on the ticket router with Unsloth on a free Colab T4, and export a GGUF.
4. **Compare** the tuned model with the 14.03 baselines and few-shot models with a paired test, on the same test set.
5. **Serve** the tuned model with Ollama on your laptop and measure latency per ticket on the CPU.
6. **Choose** a serving path (Ollama, vLLM, hosted LoRA, Smithtune) for a team, with its costs and limits.

## 📚 Official reading (verified 2026-10-16)

| Page | Read it for |
|---|---|
| [LoRA conceptual guide (PEFT)](https://huggingface.co/docs/peft/conceptual_guides/lora) | Low-rank adapters, `r`, `lora_alpha`, target modules |
| [Unsloth: fine-tuning guide](https://unsloth.ai/docs/get-started/fine-tuning-llms-guide) | QLoRA defaults, hyperparameters, training on completions only |
| [Unsloth: saving to GGUF](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf) | `save_pretrained_gguf(..., quantization_method="q4_k_m")`, chat-template pitfalls |
| [Ollama: importing a model](https://docs.ollama.com/import) | `FROM ./model.gguf` in a Modelfile, `ollama create` |
| [vLLM: LoRA adapters](https://docs.vllm.ai/en/latest/features/lora.html) | Serving many adapters on one GPU (Linux) |
| [Groq LoRA](https://console.groq.com/docs/lora) | Hosted LoRA inference (enterprise plans only) |

> **Why this matters at work.** 14.03 showed that a few-shot 4B model routes 94% of tickets right but takes about 6 minutes per 234 tickets on a laptop CPU, while a 0.6B model is fast and only 47% right. Fine-tuning a **small** model on 600 labelled tickets is how teams get "big-model accuracy at small-model cost": this lesson tests that claim on your data instead of believing it.

> 🖥️ **CPU-only laptop?** Perfect for this lesson's plan: training runs on a free **Colab T4 GPU**; your laptop only compares results (offline) and **serves** the finished model with Ollama, where a fine-tuned 0.6–1.7B model answers in well under a second per ticket on a CPU.

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-ollama>=1.1.0,<2" "numpy>=2.3.0,<3"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain-core", "langchain-ollama", "numpy",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/14-local-models-and-fine-tuning/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
LANGSMITH_PROJECT = 'm14-qlora-on-colab'
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config, look for a local Ollama.
import os

from mastery.config import env_status, is_set, load_env, load_settings
from mastery.local import ollama_status

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_HF_TOKEN = is_set("HF_TOKEN")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m14-<lesson>: one tracing project per lesson
OLLAMA = ollama_status(timeout=3.0)                              # never raises; 3 s if nothing is listening
HAVE_OLLAMA = OLLAMA.running

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("fast      :", settings.model_fast, "(Groq)")
print("local     :", settings.model_local, "| embeddings:", settings.model_local_embed)
print("ollama    :", f"{OLLAMA.version} at {OLLAMA.base_url}, models: {OLLAMA.names() or 'none pulled'}" if HAVE_OLLAMA
      else f"not running at {OLLAMA.base_url}")
if not HAVE_OLLAMA:
    print("\n⚠️ No Ollama server: 🖥️ local cells will print ⏭️ Skipped. Start the Ollama app (or `ollama serve`) and re-run.")
if not HAVE_GROQ_KEY:
    print("⚠️ No GROQ_API_KEY: 💸 Groq cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# Preview models warn once per model build; the lesson says so in Markdown already.
from mastery.models import PreviewModelWarning
warnings.filterwarnings("once", category=PreviewModelWarning)

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table (one comparison cell per Part)."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

import time

from mastery.evals.stats import paired_compare
from mastery.finetune import (QUEUES, RECORDED_DIR, SYSTEM_PROMPT, LogReg, accuracy, by_split, correct_map, load_predictions,
                              load_tickets, macro_f1)

SPLIT = by_split(load_tickets())
TRAIN, TEST = SPLIT["train"], SPLIT["test"]
GOLD = {t["id"]: t["queue"] for t in TEST}
OUT = REPO_ROOT / "data" / "outputs" / "m14"
COLAB_NB = REPO_ROOT / "modules" / "14-local-models-and-fine-tuning" / "colab" / "colab-qlora-router.ipynb"
print("test tickets:", len(TEST), "| Colab notebook:", COLAB_NB.relative_to(REPO_ROOT))

---
## Part 1 · LoRA and QLoRA in one picture

A fine-tune normally updates every weight: for a 1.7B model that's 1.7 billion numbers, plus optimizer state, about 16 bytes each. **LoRA** freezes the model and learns, for each chosen weight matrix `W` (d × k), two thin matrices `B` (d × r) and `A` (r × k) with a small rank `r` (8–64). The layer computes `W x + (alpha / r) · B A x`. Only `A` and `B` are trained: about 1–2% of the parameters. **QLoRA** also stores the frozen `W` in **4-bit**, so the base model takes a quarter of the memory.

After training you either keep the adapter separately (a few MB to ~100 MB, swap adapters per task) or **merge** `B A` into `W` and export one ordinary model (what Ollama needs: it can't load Qwen3 LoRA adapters on their own).

In [ ]:
# 🧪 Rough GPU memory for training: frozen weights + LoRA weights with Adam state + activations.
def train_memory_gb(params_b, base_bits=4, lora_fraction=0.017, activations_gb=1.5):
    base = params_b * base_bits / 8                          # GB for the frozen weights
    lora = params_b * lora_fraction * (2 + 2 + 8)            # bf16 weights + grads + Adam (2 × fp32)
    return round(base + lora + activations_gb, 1)


rows = []
for name, b in [("Qwen3-0.6B", 0.6), ("Qwen3-1.7B", 1.7), ("Qwen3-4B", 4.0), ("Qwen3-8B", 8.2), ("Qwen3-14B", 14.8)]:
    full = round(b * 16 + 1.5, 1)                            # full fine-tune: weights, grads, Adam ≈ 16 bytes/param
    q = train_memory_gb(b)
    rows.append({"model": name, "full fine-tune GB": full, "QLoRA GB": q, "free T4 (15 GB)": "✅" if q < 15 else "❌"})
table(rows, ["model", "full fine-tune GB", "QLoRA GB", "free T4 (15 GB)"])

**You should now see** full fine-tuning needing 11 GB for 0.6B up to 238 GB for 14B, while QLoRA fits up to 14B on a free T4 (the 14B row is close to the limit; Unsloth's docs confirm Qwen3-14B QLoRA on a T4 with short sequences).

### What just happened?

These are rules of thumb (activations grow with batch size and sequence length; Unsloth's gradient checkpointing shrinks them). The point is the order of magnitude: QLoRA makes 1–14B models trainable on one 15 GB GPU. On a laptop **CPU** the memory would fit for small models, but training is slow: the course measured 5.6 minutes for 2 epochs of LoRA on a 0.6B model with 594 short tickets on an 8-core CPU, against a few minutes for 1.7B on a T4. Anything bigger belongs on a GPU.

---
## Part 2 · Run the fine-tune on Colab (your turn, ≈ 30 min)

1. Open [colab.research.google.com](https://colab.research.google.com) → *File → Upload notebook* → choose `modules/14-local-models-and-fine-tuning/colab/colab-qlora-router.ipynb` from your clone.
2. *Runtime → Change runtime type → T4 GPU* → *Save*.
3. *Runtime → Run all*. Step 3 asks for a file: upload `data/samples/local/tickets.jsonl` from your clone.
4. Watch Step 5's loss fall (from about 2 to near 0) and Step 6 print the test accuracy.
5. Step 6 downloads `qlora-predictions.json`: move it to `data/outputs/m14/` in your clone.
6. Step 7 downloads the GGUF (≈ 1.1 GB for 1.7B at Q4_K_M), or pushes it to your Hub account if you set `PUSH_TO_HUB` and a `HF_TOKEN` Colab secret.
7. *Runtime → Disconnect and delete runtime* (frees your GPU quota).

What the Colab notebook does: Unsloth loads `unsloth/Qwen3-1.7B` in 4-bit, adds rank-16 LoRA adapters to all attention and MLP projections, and trains 2 epochs on the 594 training tickets with the loss **only on the answer** (`train_on_responses_only`), so the model learns to output a queue name, not to repeat tickets. The system prompt is the same `SYSTEM_PROMPT` as `mastery.finetune`.

> No GPU quota today? Colab's free T4 isn't guaranteed. Kaggle notebooks (30 GPU hours a week, phone verification) run the same notebook; the course's recorded CPU run below lets you continue either way.

---
## Part 3 · Did fine-tuning pay off? Same tickets, paired

In [ ]:
# 🧪 Every router from 14.03 plus the fine-tuned ones, on the same 234 test tickets.
lr = LogReg().fit([t["text"] for t in TRAIN], [t["queue"] for t in TRAIN])
ROUTERS = {"logistic regression": {t["id"]: p for t, (p, _) in zip(TEST, lr.predict([t["text"] for t in TEST]))}}
INFO = {}
files = [RECORDED_DIR / "fewshot-qwen3-0.6b.json", RECORDED_DIR / "fewshot-qwen3-4b.json",
         RECORDED_DIR / "lora-qwen3-0.6b-cpu.json", OUT / "qlora-predictions.json"]
for file in files:
    if file.exists():
        rec = load_predictions(file)
        ROUTERS[rec["model"]] = rec["labels"]
        INFO[rec["model"]] = rec
    else:
        print(f"(not yet: {file.name}, run the Colab notebook and put it in data/outputs/m14/)")
rows = []
for name, pred in ROUTERS.items():
    rec = INFO.get(name, {})
    rows.append({"router": name, "accuracy [95% CI]": f"{accuracy(pred, GOLD):.1%}".split(" (")[0],
                 "macro-F1": f"{macro_f1(pred, GOLD, QUEUES):.2f}",
                 "Spanish": f"{sum(pred.get(t['id']) == t['queue'] for t in TEST if t['lang'] == 'es') / 54:.0%}",
                 "recorded on": rec.get("hardware", "this notebook")})
table(rows, ["router", "accuracy [95% CI]", "macro-F1", "Spanish", "recorded on"])

**You should now see** five rows (six after your Colab run): logistic regression ≈ 63%, few-shot qwen3:0.6b ≈ 47%, few-shot qwen3:4b ≈ 94%, and the course's **LoRA-tuned Qwen3-0.6B ≈ 87%** (and 100% of the Spanish tickets), recorded on a CPU. Your `qlora:unsloth/Qwen3-1.7B` row usually lands between 88% and 96%.

### What just happened?

The same 0.6B model went from 47% (few-shot) to 87% after 5.6 minutes of LoRA training on 594 examples: fine-tuning taught it the label scheme that the prompt alone couldn't. Spanish jumped to 100% because the training set has one Spanish family per queue (the recorded file states how it was trained: LoRA r=16 on all linear layers, 2 epochs, CPU). It still trails the few-shot 4B model; a tuned 1.7B (your Colab run) often closes the gap.

In [ ]:
# 🧪 Paired: is the tuned model really better than its own base, and than the big few-shot model?
tuned = "lora:Qwen/Qwen3-0.6B (r=16, 2 epochs)"
pairs = [("ollama:qwen3:0.6b", tuned), ("logistic regression", tuned), ("ollama:qwen3:4b", tuned)]
mine = next((n for n in ROUTERS if n.startswith("qlora:")), None)
if mine:
    pairs += [("ollama:qwen3:4b", mine), (tuned, mine)]
for a, b in pairs:
    r = paired_compare(correct_map(ROUTERS[a], GOLD), correct_map(ROUTERS[b], GOLD), names=(a, b))
    print(f"{b} vs {a}: {r.b_wins} wins, {r.a_wins} losses, diff {r.mean_diff.estimate:+.1%}, p = {r.p_value:.2g} → {r.verdict()}")

**You should now see** the tuned 0.6B clearly better than its few-shot base (+40 points, tiny p) and than logistic regression, and **worse** than few-shot qwen3:4b (about −8 points, p < 0.05). With your Colab file, two more lines compare your 1.7B with the 4B and with the CPU run.

### What just happened?

Fine-tuning is worth it here when the **constraint is the model size**: a 1.2 GB model at 87% versus a 2.5 GB model at 94% that is many times slower per ticket. If the laptop can afford the 4B model's latency, few-shot is the simpler system (no training pipeline to maintain). Part 4 measures the latency side.

---
## Part 4 · Serve the tuned model with Ollama

> 🖥️ Runs **on your computer** through Ollama: no key, no cost, nothing leaves the machine. On a **CPU-only** laptop (no NVIDIA GPU) expect 10–60 s for the first call (the model loads into RAM) and roughly 5–15 tokens per second for a 4B model afterwards; the course gives local calls a 300 s timeout. Without a running Ollama it prints `⏭️ Skipped`.

In a terminal, in the folder where you saved the GGUF from Colab (file name as downloaded), create a `Modelfile`:

```text
FROM ./orbita-router.Q4_K_M.gguf
PARAMETER temperature 0
PARAMETER num_predict 8
```

then:

```powershell
ollama create orbita-router -f Modelfile
ollama run orbita-router "I was charged twice for October."
```

Ollama reads the chat template from the GGUF's metadata (it was written by Unsloth from the Qwen3 tokenizer); that's why serving must use the **same system prompt** as training. If you pushed to the Hub, `ollama pull hf.co/<you>/orbita-router-GGUF:Q4_K_M` instead, then `ollama cp` it to `orbita-router`.

In [ ]:
# 🖥️ (HAVE_OLLAMA) The served router: 40 test tickets one at a time, accuracy and seconds per ticket.
from mastery.models import get_model

TUNED = "orbita-router"        # ← the name you gave `ollama create`
if HAVE_OLLAMA and OLLAMA.get(TUNED):
    served = get_model(model=f"ollama:{TUNED}", temperature=0, max_tokens=8)
    start, labels = time.perf_counter(), {}
    for t in TEST[:40]:
        words = served.invoke([("system", SYSTEM_PROMPT), ("user", t["text"])]).text.split()
        labels[t["id"]] = words[0] if words and words[0] in QUEUES else "error"
    seconds = time.perf_counter() - start
    sub = {i: GOLD[i] for i in labels}
    print(f"{TUNED} ({OLLAMA.get(TUNED).parameters}, {OLLAMA.get(TUNED).quantization}): "
          f"{accuracy(labels, sub).estimate:.0%} on 40 tickets, {seconds / 40:.2f} s per ticket on this machine")
    print(f"recorded few-shot qwen3:4b on the same 40: {accuracy(ROUTERS['ollama:qwen3:4b'], sub).estimate:.0%}, "
          f"≈ {INFO['ollama:qwen3:4b']['seconds'] / len(TEST):.2f} s per ticket (batched, recorded)")
elif HAVE_OLLAMA:
    print(f"⏭️ Skipped: no '{TUNED}' model in Ollama yet (ollama create {TUNED} -f Modelfile, see above).")
else:
    print("⏭️ Skipped: no Ollama server (see the setup cell).")

**You should now see**, after `ollama create`, a line such as `orbita-router (1.7B, Q4_K_M): 90% on 40 tickets, 0.3 s per ticket`, against about 1.6 s per ticket for the few-shot 4B model, or `⏭️ Skipped`. (The course's CPU-trained 0.6B, served at F16: 88% on these 40, 0.16 s per ticket on an 8-core CPU, against 95% for the recorded 4B.)

### What just happened?

The tuned model needs no examples in the prompt and answers with one or two tokens, so it's fast even on a CPU: prompt processing is short and generation stops at once (`num_predict 8`). That's the real payoff of fine-tuning a narrow task: **shorter prompts, smaller model, same machine**.

---
## Part 5 · Serving options for a team

| Option | Runs where | LoRA support | Cost | Good for |
|---|---|---|---|---|
| **Ollama + merged GGUF** (this lesson) | Laptop, any OS, CPU or GPU | Merge first (Qwen3 adapters can't be loaded alone) | Free | One user, privacy, prototypes |
| **vLLM** | Linux + NVIDIA GPU (not native Windows) | Many adapters on one base, per request (`--enable-lora`) | A GPU server | A team, many tasks, throughput |
| **llama.cpp server** | Anywhere | Adapters as GGUF (`--lora`) | Free | Lightweight self-hosting |
| **Groq LoRA** | Hosted | Enterprise plans only; base `llama-3.1-8b-instant`; inference only | Contract | Teams already on Groq enterprise |
| **Hugging Face Inference Endpoints** | Hosted, dedicated GPU | Merged models or adapters | Per hour of GPU | A private API without running servers |
| **LangSmith Smithtune** (beta) | Hosted (Fireworks/Baseten) | Trains from your LangSmith traces, compares base vs tuned in an experiment | Paid providers | Teams whose data already lives in LangSmith |

The course's choice for Orbita Cloud: train on Colab, serve the merged GGUF with Ollama on the support laptops, keep the test set and the 14.03 comparison as the release gate for every retrain.

---
## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| Colab: `No GPU` / `CUDA not available` | Runtime is CPU | *Runtime → Change runtime type → T4 GPU*, then *Run all* again |
| `OutOfMemoryError` on the T4 | Batch or sequence too large, or a big model | `per_device_train_batch_size=4`, `max_seq_length=256`, or the 1.7B model |
| Accuracy far below Colab after `ollama create` | Different prompt or chat template at serving time | Use the same `SYSTEM_PROMPT`; check `ollama show orbita-router --modelfile` |
| Output like `<think>` or several words | Thinking mode or missing stop token | `num_predict 8`, `temperature 0`; parse the first word (as the cell does) |
| `create-time quantization is only supported for safetensors imports` | `ollama create -q` on a GGUF | Quantize when exporting (`quantization_method="q4_k_m"` in Unsloth), import the GGUF as is |
| Old Ollama rejects the GGUF | Ollama older than the model architecture | Update Ollama |
| Loss drops to 0.000 | The answer is one or two tokens and the train set is small | Expected; watch **test** accuracy, not loss (the course trains 2 epochs, not more) |

## ⚠️ Production notes

> ⚠️ **Production note: a fine-tune is a release.** (Parts 3–4)
> Version the dataset, the training notebook and the GGUF together; re-run the 14.03 comparison on the frozen test set before shipping, and keep the previous model to roll back.

> ⚠️ **Production note: the tuned model inherits the base licence.** (Part 2)
> Qwen3 is Apache-2.0, so the tuned router is too; a Llama or Gemma base would bring its custom terms. Training data with customer tickets is personal data: anonymize it before it leaves the company network for Colab.

## 🏋️ Exercises

1. **(Easy)** In the Colab notebook set `BASE = "unsloth/Qwen3-4B"` and run again. Is the gain over 1.7B worth twice the size?
2. **(Medium)** Train for 1 epoch instead of 2 (`num_train_epochs=1`). Compare test accuracy with a paired test against your 2-epoch run.
3. **(Medium)** Add the 72 dev tickets to training on Colab (`t["split"] in ("train", "dev")`). Do Spanish tickets improve? Why or why not?
4. **(Stretch)** Serve both `orbita-router` and `qwen3:4b` and build the 14.03 two-tier router with the tuned model as tier 1 (no confidence: use "answer not in QUEUES" or a second run disagreeing as the escalation signal).

<details>
<summary><b>Show solutions</b></summary>

**1.** Change `BASE`, rerun Steps 4–7; a 4B QLoRA usually adds a few points (often to 95–97%) at about 2.5 GB and roughly half the speed on a CPU. Decide with Part 4's seconds per ticket.

**2.** `paired_compare(correct_map(one_epoch, GOLD), correct_map(two_epochs, GOLD))`; with 234 tickets differences under ~5 points are rarely significant.

**3.** Dev has no Spanish tickets (14.03, Part 6), so Spanish accuracy barely moves: more data helps only where it adds new patterns.

**4.** For each ticket, ask `orbita-router`; if the answer isn't a queue, or `temperature=0.7` twice gives different answers, send it to `qwen3:4b`. Count escalations and accuracy with `correct_map`.
</details>

## 🔎 Docs drill (8 min)

In PEFT's LoRA guide, what do `r` and `lora_alpha` control, and what does Unsloth recommend as a starting point?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Open the [LoRA conceptual guide](https://huggingface.co/docs/peft/conceptual_guides/lora): `r` is the rank of the update matrices (capacity and size); the update is scaled by `lora_alpha / r`.
2. Open the [Unsloth fine-tuning guide](https://unsloth.ai/docs/get-started/fine-tuning-llms-guide): start with `r = 16` and `lora_alpha = r` (or `2 × r`), all linear projections as target modules.
3. The Colab notebook uses exactly `r=16, lora_alpha=16`.

</details>

## ✅ Checkpoint

- [ ] I can explain LoRA, QLoRA and merging in two sentences each.
- [ ] I ran the Colab notebook (or know why I used the recorded CPU run) and have `qlora-predictions.json`.
- [ ] I compared the tuned model with its base and with few-shot 4B using a paired test.
- [ ] `orbita-router` runs in Ollama on my laptop and I measured seconds per ticket.
- [ ] I can say which serving option fits a one-person tool and which fits a team.

## 🔎 Research checkpoint

Log in [`docs/research-log.md`](../../docs/research-log.md): your Colab GPU, training seconds, test accuracy of the tuned model, its GGUF size, and seconds per ticket on your laptop. Then check the current [Unsloth Qwen3 page](https://unsloth.ai/docs) for the largest Qwen model it trains on a free T4.

## Summary

- LoRA trains ~1–2% of the parameters as low-rank adapters; QLoRA keeps the frozen base in 4-bit, so 1–14B models train on a free T4.
- Train on Colab, merge, export a Q4_K_M GGUF, `ollama create` it: the laptop only serves.
- On the router, LoRA lifted Qwen3-0.6B from 47% to 87%; few-shot 4B still leads at 94%, at many times the cost per ticket.
- Fine-tuning pays when the constraint is size or latency on a narrow, stable task; measure it with the same frozen test set.
- Teams serve adapters with vLLM or hosted services; one laptop is fine with Ollama.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: the **module project**: read [project/README.md](project/README.md) (spec, modes, acceptance criteria).

[Module 14 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)